# **Chuẩn bị datasets**

In [ ]:
# 1. CÀI ĐẶT KAGGLE VÀ UPLOAD API KEY
!pip install -q kaggle
from google.colab import files
print("Vui lòng upload file kaggle.json của bạn:")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. TẢI DATASET GỐC (36 Loại)
print("Đang tải dataset gốc...")
!kaggle datasets download -d kritikseth/fruit-and-vegetable-image-recognition
!unzip -q fruit-and-vegetable-image-recognition.zip -d dataset

# 3. TẢI DATASET MỞ RỘNG (Fruits 360)
print("Đang tải dataset Fruits 360...")
!kaggle datasets download -d moltean/fruits
!unzip -q fruits.zip -d fruits_360_temp

print("✅ Tải và giải nén hoàn tất!")

Vui lòng upload file kaggle.json của bạn:


Saving kaggle.json to kaggle.json
Đang tải dataset gốc...
Dataset URL: https://www.kaggle.com/datasets/kritikseth/fruit-and-vegetable-image-recognition
License(s): CC0-1.0
100% 1.98G/1.98G [01:52<00:00, 18.9MB/s]

Đang tải dataset Fruits 360...
Dataset URL: https://www.kaggle.com/datasets/moltean/fruits
License(s): CC-BY-SA-4.0
 87% 5.23G/5.99G [05:09<00:48, 17.0MB/s]

In [ ]:
!rm -rf dataset
print("Đang tải lại dataset gốc...")
!unzip -q fruit-and-vegetable-image-recognition.zip -d dataset

# **Lọc và Trộn Dataset (Bơm dữ liệu mới vào dữ liệu cũ)**
Ô code này sẽ tự động nhặt các ảnh sắc nét từ Fruits 360 và thả đúng vào 36 thư mục train của bạn.

In [ ]:
import os
import shutil

CLASS_NAMES = ['apple', 'banana', 'beetroot', 'bell pepper', 'cabbage', 'capsicum', 'carrot', 'cauliflower', 'chilli pepper', 'corn', 'cucumber', 'eggplant', 'garlic', 'ginger', 'grapes', 'jalepeno', 'kiwi', 'lemon', 'lettuce', 'mango', 'onion', 'orange', 'paprika', 'pear', 'peas', 'pineapple', 'pomegranate', 'potato', 'raddish', 'soy beans', 'spinach', 'sweetcorn', 'sweetpotato', 'tomato', 'turnip', 'watermelon']

# 🌟 TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN CHUẨN
base_dir = '/content/fruits_360_temp'
src_dir = None

# Quét tìm thư mục có chứa chữ "original" và thư mục "Training" bên trong
for root, dirs, files in os.walk(base_dir):
    if 'Training' in dirs and 'original' in root.lower():
        src_dir = os.path.join(root, 'Training')
        break

if src_dir is None:
    raise FileNotFoundError("⚠️ Không tìm thấy thư mục Training của nhánh original-size. Hãy kiểm tra lại file unzip!")

print(f"✅ Đã tìm thấy thư mục gốc chuẩn: {src_dir}")

dest_dir = 'dataset/train'

# CHIẾN THUẬT MỚI: Bơm tối đa 300 ảnh mỗi loại
MAX_ADD_PER_CLASS = 300
added_counts = {c: 0 for c in CLASS_NAMES}
copied_count = 0

print("🚀 Bắt đầu lọc và gộp dữ liệu (Luật ánh xạ thông minh)...")

for folder_name in os.listdir(src_dir):
    src_folder_path = os.path.join(src_dir, folder_name)
    if not os.path.isdir(src_folder_path): continue

    folder_lower = folder_name.lower()

    for target_class in CLASS_NAMES:
        match = False # Biến cờ hiệu xác định xem có đúng trọng tâm không

        # --- 1. LUẬT CẤM (CHỐNG NHẬN NHẦM) ---
        if target_class == 'apple' and 'pineapple' in folder_lower: continue
        if target_class == 'corn' and 'sweetcorn' in folder_lower: continue
        if target_class == 'orange' and ('tomato' in folder_lower or 'pepper' in folder_lower): continue
        if target_class == 'potato' and 'sweet' in folder_lower: continue # Cấm khoai lang lọt vào khoai tây
        if target_class == 'lemon' and 'melon' in folder_lower: continue # Tránh nhầm dưa hấu/dưa lưới với chanh

        # --- 2. LUẬT KHỚP TÊN ĐẶC BIỆT (XỬ LÝ SAI LỆCH GIỮA 2 DATASET) ---
        if target_class == 'grapes' and 'grape' in folder_lower and 'grapefruit' not in folder_lower:
            match = True # Fruits-360 gọi 'grape', mình map nó về 'grapes'
        elif target_class == 'sweetpotato' and 'potato sweet' in folder_lower:
            match = True # Map 'potato sweet' về class 'sweetpotato'
        elif target_class == 'bell pepper' and 'pepper' in folder_lower and 'orange' not in folder_lower:
            match = True # Gom Pepper Red/Green/Yellow về 'bell pepper'
        elif target_class == 'eggplant' and ('aubergine' in folder_lower or 'eggplant' in folder_lower):
            match = True

        # --- 3. LUẬT TÌM KIẾM BÌNH THƯỜNG ---
        elif target_class in folder_lower:
            match = True

        # --- NẾU ĐÚNG TRỌNG TÂM THÌ TIẾN HÀNH GỘP ---
        if match:
            dest_folder_path = os.path.join(dest_dir, target_class)
            os.makedirs(dest_folder_path, exist_ok=True)

            for img_file in os.listdir(src_folder_path):
                # Kiểm tra nếu đã bơm đủ 300 ảnh thì dừng lại
                if added_counts[target_class] >= MAX_ADD_PER_CLASS:
                    break

                src_img = os.path.join(src_folder_path, img_file)
                dest_img = os.path.join(dest_folder_path, f"f360_original_{folder_name}_{img_file}")

                if not os.path.exists(dest_img):
                    shutil.copy(src_img, dest_img)
                    added_counts[target_class] += 1
                    copied_count += 1

            print(f"  + Đã gộp [{folder_name}] vào class [{target_class}] - Tổng đã thêm: {added_counts[target_class]}/{MAX_ADD_PER_CLASS}")
            break # Tìm đúng rồi thì thoát vòng lặp class, qua thư mục mới

print(f"\n🎉 Hoàn tất! Đã bơm thêm {copied_count} bức ảnh sắc nét vào tập Train.")

# **Cấu hình hệ thống (Config), Data Augmentation & DataLoader**
Ép model nhìn ảnh kích thước 128x128 và thêm nhiễu mạnh (Augmentation) để chống học vẹt.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms
import warnings
warnings.filterwarnings('ignore')

# --- CẤU HÌNH V2 (GIỮ NGUYÊN) ---
CONFIG = {
    'data_dir'      : './dataset',
    'img_size'      : 128,
    'batch_size'    : 64,
    'num_epochs'    : 50,
    'learning_rate' : 0.001,
    'num_workers'   : 2,
    'device'        : 'cuda' if torch.cuda.is_available() else 'cpu',
}
device = torch.device(CONFIG['device'])
print(f'⚙️ Cấu hình V2 Loaded | Chạy trên: {CONFIG["device"].upper()}')

# --- DATA AUGMENTATION (TẬP TRAIN) ---
train_transforms = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15))
])

# --- TRANSFORMS (TẬP VALIDATION) ---
val_transforms = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),           # Đảm bảo ảnh test cũng không bị méo
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# =====================================================================
# 🌟 CHIẾN THUẬT CHIA 80 - 20 TỰ ĐỘNG BẰNG PYTORCH
# =====================================================================

# 1. Đọc toàn bộ ảnh từ cả 2 thư mục train và test hiện tại mà không áp dụng transform trước
raw_train_dataset = datasets.ImageFolder(f"{CONFIG['data_dir']}/train")
raw_test_dataset  = datasets.ImageFolder(f"{CONFIG['data_dir']}/test")

# 2. Gộp tất cả ảnh lại thành một rổ dữ liệu tổng khổng lồ
full_dataset = ConcatDataset([raw_train_dataset, raw_test_dataset])
total_images = len(full_dataset)

# 3. Tính toán số lượng để chia chính xác tỷ lệ 80% - 20%
train_size = int(0.8 * total_images)
val_size = total_images - train_size

# 4. Tiến hành phân tách ngẫu nhiên các chỉ số (indices) ảnh
train_subset, val_subset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# 5. Gán chính xác transform riêng cho từng tập (Train thì biến đổi, Val thì giữ nguyên để test)
class WrappedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            # Lấy ảnh gốc ra và ép lại transform chuẩn
            raw_img = self.subset.dataset[self.subset.indices[index]][0]
            x = self.transform(raw_img)
        return x, y
    def __len__(self):
        return len(self.subset)

# Tạo Dataset hoàn chỉnh sau khi phân tách
final_train_dataset = WrappedDataset(train_subset, train_transforms)
final_val_dataset   = WrappedDataset(val_subset, val_transforms)

# 6. ĐƯA VÀO DATALOADER ĐỂ SẴN SÀNG TRAIN
train_loader = DataLoader(final_train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=CONFIG['num_workers'])
val_loader   = DataLoader(final_val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])

# Lấy danh sách class chuẩn từ dataset gốc
class_names = raw_train_dataset.classes
num_classes = len(class_names)

print(f'📊 Tổng số ảnh trong hệ thống: {total_images:,}')
print(f'🔥 Số ảnh dùng để Train (80%): {len(final_train_dataset):,}')
print(f'🧪 Số ảnh dùng để Validation (20%): {len(final_val_dataset):,}')
print(f'🎯 Số lớp thực tế được cấu hình: {num_classes}')

# **Xem trước ảnh đã được Data Augmentation (Chạy sau khi tạo DataLoader)**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torchvision

def imshow(inp, title=None):
    """Hàm giải chuẩn hóa và hiển thị tensor ảnh."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1) # Giới hạn giá trị pixel từ 0-1
    plt.imshow(inp)
    if title is not None:
        plt.title(title, fontsize=10, fontweight='bold')
    plt.axis('off')

# Lấy 1 batch dữ liệu ra để xem thử
inputs, classes = next(iter(train_loader))

# SỬA Ở ĐÂY: Lấy tên class từ raw_train_dataset
class_names = raw_train_dataset.classes

# Lấy 8 ảnh đầu tiên trong batch
out = torchvision.utils.make_grid(inputs[:8], nrow=4)

plt.figure(figsize=(15, 6))
imshow(out, title=" | ".join([class_names[x] for x in classes[:8]]))
plt.suptitle("Preview: Ảnh sau khi được Data Augmentation (Tăng cường dữ liệu)", fontsize=14)
plt.show()

# **Kiến trúc Mạng CNN Tự Build (FruitCNN)**

In [ ]:
import torch.nn as nn

class FruitCNN_V3(nn.Module):
    def __init__(self, num_classes):
        super(FruitCNN_V3, self).__init__()

        # Khối 1: 3 -> 32 channels
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
        )
        # Khối 2: 32 -> 64 channels
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
        )
        # Khối 3: 64 -> 128 channels
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
        )
        # 🌟 THÊM MỚI - Khối 4: 128 -> 256 channels (Học đặc trưng sâu hơn)
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.3),
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512), nn.LeakyReLU(0.1, inplace=True), nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

model = FruitCNN_V3(num_classes=num_classes).to(device)
print(f'🧠 Đã khởi tạo model FruitCNN_V3 với {sum(p.numel() for p in model.parameters()):,} tham số.')

# **Huấn luyện (Training Loop) & Tải Model**
Sử dụng AdamW và Scheduler để đạt hiệu suất cao nhất. Model tốt nhất sẽ tự tải về máy sau khi chạy xong.

In [ ]:
import time
import copy
import torch

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

# 🌟 THÊM BIẾN HISTORY ĐỂ LƯU DỮ LIỆU VẼ BIỂU ĐỒ
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print("🚀 Bắt đầu huấn luyện...")
print("-" * 50)

for epoch in range(CONFIG['num_epochs']):
    start_time = time.time()

    # --- TRAIN ---
    model.train()
    train_loss, train_correct = 0.0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)

    # SỬA LỖI Ở ĐÂY: Dùng final_train_dataset
    epoch_train_loss = train_loss / len(final_train_dataset)
    epoch_train_acc = (train_correct.double() / len(final_train_dataset)) * 100

    # --- VALIDATION ---
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    # SỬA LỖI Ở ĐÂY: Dùng final_val_dataset
    epoch_val_loss = val_loss / len(final_val_dataset)
    epoch_val_acc = (val_correct.double() / len(final_val_dataset)) * 100

    # 🌟 GHI CHÉP LẠI LỊCH SỬ CHO EPOCH NÀY
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_acc'].append(epoch_train_acc.item())
    history['val_acc'].append(epoch_val_acc.item())

    scheduler.step(epoch_val_acc)
    epoch_mins, epoch_secs = divmod(time.time() - start_time, 60)

    print(f"Epoch {epoch+1:02d}/{CONFIG['num_epochs']} | {int(epoch_mins)}m {int(epoch_secs)}s")
    print(f"  Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
    print(f"  Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.2f}%")

    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'best_model_v5.pth')
        print(f"  ⭐ Đã lưu model tốt nhất (Acc: {best_acc:.2f}%)")
    print("-" * 50)
print(f"🎉 Hoàn tất! Accuracy cao nhất: {best_acc:.2f}%")

from google.colab import files
files.download('best_model_v5.pth')

# **Vẽ biểu đồ Accuracy và Loss (Chạy sau khi Train xong)**

In [ ]:
# Đảm bảo bạn đã chạy xong Ô Code 5 ở trên
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid") # Làm biểu đồ đẹp và chuyên nghiệp hơn

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị 1: ACCURACY
ax1.plot(history['train_acc'], label='Train Accuracy', marker='o', color='blue')
ax1.plot(history['val_acc'], label='Validation Accuracy', marker='o', color='green')
ax1.set_title('Biểu đồ Accuracy (Độ chính xác) qua các Epoch', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Accuracy (%)')
ax1.legend()

# Đồ thị 2: LOSS
ax2.plot(history['train_loss'], label='Train Loss', marker='s', color='red')
ax2.plot(history['val_loss'], label='Validation Loss', marker='s', color='orange')
ax2.set_title('Biểu đồ Loss (Độ lỗi) qua các Epoch', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()